# TB Portals — Spatial ALP Heatmaps (Figure F2)
Loads the saved `SpatialALPHead` from the SG-ALP (a1) mode, renders per-patch ALP involvement overlays on 6 representative cases (1 mild + 1 severe per country).

Attach: `tb-portals-cxr-pngs`. Internet **ON**. GPU T4. Runtime ≈ 10 min.

In [1]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'cleaned-repo'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'], check=False)
print('ready')

Cloning into '/kaggle/working/dl-project-codebase'...
Updating files: 100% (500/500), done.


ready


In [2]:
import os, sys, pandas as pd
from pathlib import Path
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
FEATURES_GRID = f'{WORK}/features_rad-dino_grid7.npz'
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')

from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('manifest:', len(paper_df))

from cache_features import main as cache_main, load_features
if not os.path.isfile(FEATURES_GRID):
    cache_main(['--manifest', PAPER_MANIFEST, '--out', FEATURES_GRID,
                '--backbone', 'rad-dino', '--patch-grid', '7', '--batch-size', '32'])
print('grid features ready')

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
manifest: 5010
[cache] backbone=rad-dino device=cuda patch_grid=7


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote features (5010, 49, 768) -> /kaggle/working/features_rad-dino_grid7.npz
grid features ready


In [3]:
# Train SG-ALP R1 head briefly per country (seed 0), save heads
from src.training.train_agentic import main as agentic_main
outdir = f'{WORK}/alpspatial_run'
if not os.path.isdir(outdir):
    agentic_main(['--features', FEATURES_GRID, '--manifest', PAPER_MANIFEST,
                  '--mode', 'a1', '--out-dir', outdir,
                  '--rungs', '1', '--seeds', '0',
                  '--held-outs', 'Romania', 'Moldova', 'Kazakhstan',
                  '--save-heads'])
print('training done')

[agentic] device=cuda mode=a1 dim=768 feat=grid grid_available=True rungs=[1] seeds=[0] M=5
[agentic] 6 configs: ['rung1_mse', 'rung1_bmc', 'agentic_best', 'rung4b_spatial_cavity', 'agentic_best_spatcav', 'stacked_legacy']
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[a1|rung1_mse] Romania s0  MAE=19.43 CI[17.4,21.6] r=0.650 a=0.00 | base 26.84 paper 23.83 -> BEATS
[tbportals] split held_out=Moldova: train=3531 val=890 test=589 (train/val patients 3282/820).
[a1|rung1_mse] Moldova s0  MAE=23.25 CI[21.5,25.1] r=0.796 a=0.00 | base 32.76 paper 24.44 -> BEATS
[tbportals] split held_out=Kazakhstan: train=3690 val=921 test=399 (train/val patients 3434/858).
[a1|rung1_mse] Kazakhstan s0  MAE=20.04 CI[18.2,22.1] r=0.666 a=0.00 | base 21.87 paper 22.13 -> within-noise
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[a1|rung1_bmc] Romania s0  MAE=19.54 CI[17.3,21.9] r=0.658 a=0.00 | base 26.84 paper

In [4]:
import torch, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from src.components.feature_heads import SpatialALPHead
from src.data.tbportals import make_country_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
grid_feats, dim = load_features(FEATURES_GRID)
manifest = pd.read_csv(PAPER_MANIFEST, dtype={'image_id': str})
id_to_path = dict(zip(manifest['image_id'], manifest['image_path']))
id_to_alp  = dict(zip(manifest['image_id'], manifest['alp_0_100']))
id_to_cav  = dict(zip(manifest['image_id'], manifest['cavity']))

OUT_DIR = Path(f'{WORK}/alp_spatial_pngs')
OUT_DIR.mkdir(exist_ok=True)

def overlay(image_path, cell49, title, out_path):
    img = np.asarray(Image.open(image_path).convert('L'), dtype=np.float32) / 255.0
    H, W = img.shape
    grid = cell49.reshape(7, 7)
    fig, ax = plt.subplots(figsize=(3.2, 3.4))
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.imshow(grid, extent=(0, W, H, 0), alpha=0.55, cmap='magma',
              vmin=0, vmax=1, interpolation='bilinear')
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(pad=0.2)
    plt.savefig(out_path, dpi=140, bbox_inches='tight'); plt.close()

rows = []
for country in ['Romania', 'Moldova', 'Kazakhstan']:
    head_path = Path(outdir) / 'heads' / f'a1_rung1_bmc_{country}_s0.pt'
    if not head_path.exists():
        # try the mse rung name if bmc not saved
        head_path = Path(outdir) / 'heads' / f'a1_rung1_mse_{country}_s0.pt'
    blob = torch.load(head_path, map_location=device)
    head = SpatialALPHead(in_dim=dim).to(device).eval()
    head.load_state_dict(blob['reg_heads'][0])

    _, _, test_df = make_country_split(manifest, held_out_country=country, val_fraction=0.2, seed=0)
    test_ids = [i for i in test_df['image_id'].astype(str).values if i in grid_feats]
    alps = [(i, float(id_to_alp.get(i, 0))) for i in test_ids]
    mild  = sorted([(i, a) for i, a in alps if a < 30], key=lambda t: t[1])[:1]
    severe= sorted([(i, a) for i, a in alps if a > 70], key=lambda t: -t[1])[:1]
    chosen = [(mild[0][0], 'mild'), (severe[0][0], 'severe')] if mild and severe else []
    for img_id, tag in chosen:
        Xg = torch.from_numpy(grid_feats[img_id][None]).float().to(device)
        with torch.no_grad():
            alp_pred, cell = head(Xg, return_map=True)
            cell49 = cell[0].cpu().numpy()
            alp_pred_val = float(alp_pred[0].cpu()) * 100
        title = f'{country} {tag}: GT ALP={id_to_alp[img_id]:.0f} | pred={alp_pred_val:.0f}'
        png_path = OUT_DIR / f'{country}_{tag}_{img_id[-8:]}.png'
        overlay(id_to_path[img_id], cell49, title, png_path)
        rows.append({'country': country, 'tag': tag, 'image_id': img_id,
                     'gt_alp': id_to_alp[img_id], 'pred_alp': alp_pred_val, 'png': str(png_path)})
    print(f'{country}: 2 PNGs')

pd.DataFrame(rows).to_csv(OUT_DIR / 'meta.csv', index=False)
import shutil
zip_path = shutil.make_archive(f'{WORK}/alp_spatial_pngs', 'zip', str(OUT_DIR))
print('zip ->', zip_path)

[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
Romania: 2 PNGs
[tbportals] split held_out=Moldova: train=3531 val=890 test=589 (train/val patients 3282/820).
Moldova: 2 PNGs
[tbportals] split held_out=Kazakhstan: train=3690 val=921 test=399 (train/val patients 3434/858).
Kazakhstan: 2 PNGs
zip -> /kaggle/working/alp_spatial_pngs.zip
